In [ ]:
# Install geospatial libraries
!pip install geopandas rasterio fiona requests -q
# Test the BC Catalogue API immediately
import requests

response = requests.get(
    "https://catalogue.data.gov.bc.ca/api/3/action/package_search",
    params={"q": "cadastral survey", "rows": 5}
)

data = response.json()
print(f"Success: {data['success']}")
print(f"Total datasets found: {data['result']['count']}")

for dataset in data['result']['results']:
    print(f"\n{dataset['title']}")
    formats = [r['format'] for r in dataset.get('resources', [])]
    print(f"Formats: {formats}")

Success: True
Total datasets found: 35

North Cowichan Cadastral
Formats: ['kmz', 'csv', 'other', 'fgdb', 'gpkg', 'shp']

TANTALIS - Surveyed Parcels
Formats: ['multiple', 'wms', 'kml']

TANTALIS - Integrated Survey Areas
Formats: ['multiple', 'wms', 'kml']

TANTALIS - Surveyed Wellsites
Formats: ['multiple', 'wms', 'kml']

TANTALIS - Surveyed Parcels (Polygons)
Formats: ['oracle_sde']


In [ ]:
import json

# Grab the first dataset from your previous results
first_dataset = data['result']['results'][0]

# Pretty print the entire thing so you can read it
print(json.dumps(first_dataset, indent=2))

{
  "author": "ab613df5-a7ff-4c3e-a215-639af6f4227b",
  "author_email": null,
  "creator_user_id": "ab613df5-a7ff-4c3e-a215-639af6f4227b",
  "data_quality": "Cadastral features are updated as parcels are surveyed, subdivided, or created during municipal processes.",
  "download_audience": "Public",
  "id": "17d1b2cf-2f7e-4928-b337-6768e286a852",
  "isopen": false,
  "license_id": "44",
  "license_title": "Open Government Licence \u2013 Municipality of North Cowichan",
  "license_url": "https://www.northcowichan.ca/EN/footer/open-data-licence.html",
  "lineage_statement": "See Data Quality.",
  "maintainer": null,
  "maintainer_email": null,
  "metadata_created": "2018-10-04T18:21:27.718131",
  "metadata_modified": "2022-04-26T20:53:11.923340",
  "metadata_visibility": "Public",
  "name": "north-cowichan-cadastral",
  "notes": "Cadastral dataset containing; parcels, parcel plus (addressing), easements & covenants, Section/Range/Land District, Strata parcels, and the North Cowichan bound

In [ ]:
print(f"Dataset: {first_dataset['title']}\n")
print("Available resources:")

for i, resource in enumerate(first_dataset.get('resources', [])):
    print(f"\n  Resource {i+1}:")
    print(f"    Name:   {resource.get('name', 'unnamed')}")
    print(f"    Format: {resource.get('format', 'unknown')}")
    print(f"    URL:    {resource.get('url', 'no url')}")

Dataset: North Cowichan Cadastral

Available resources:

  Resource 1:
    Name:   Cadastral KMZ
    Format: kmz
    URL:    https://s3-us-west-2.amazonaws.com/openfiles.northcowichan.ca/GIS/Cadastral/Cadastral.kmz

  Resource 2:
    Name:   Cadastral CSV
    Format: csv
    URL:    https://s3-us-west-2.amazonaws.com/openfiles.northcowichan.ca/GIS/Cadastral/Cadastral_CSV.zip

  Resource 3:
    Name:   Cadastral DWG
    Format: other
    URL:    https://s3-us-west-2.amazonaws.com/openfiles.northcowichan.ca/GIS/Cadastral/Cadastral_DWG.zip

  Resource 4:
    Name:   Cadastral FGDB
    Format: fgdb
    URL:    https://s3-us-west-2.amazonaws.com/openfiles.northcowichan.ca/GIS/Cadastral/Cadastral_FGDB.zip

  Resource 5:
    Name:   Cadastral GeoPackage
    Format: gpkg
    URL:    https://s3-us-west-2.amazonaws.com/openfiles.northcowichan.ca/GIS/Cadastral/Cadastral_GEO.zip

  Resource 6:
    Name:   Cadastral SHP
    Format: shp
    URL:    https://s3-us-west-2.amazonaws.com/openfiles.northc

In [ ]:
from collections import Counter

# Pull more results
response2 = requests.get(
    "https://catalogue.data.gov.bc.ca/api/3/action/package_search",
    params={"q": "cadastral survey", "rows": 50}
)
data2 = response2.json()
datasets = data2['result']['results']

# Count all format types across all resources
all_formats = []
for dataset in datasets:
    for resource in dataset.get('resources', []):
        fmt = resource.get('format', 'UNKNOWN').upper()
        if fmt:
            all_formats.append(fmt)

format_counts = Counter(all_formats)
print("Format types found across 50 datasets:\n")
for fmt, count in format_counts.most_common():
    print(f"  {fmt:<20} {count} resources")


Format types found across 50 datasets:

  MULTIPLE             28 resources
  WMS                  17 resources
  KML                  17 resources
  ORACLE_SDE           6 resources
  FGDB                 4 resources
  PDF                  2 resources
  XLSX                 2 resources
  KMZ                  1 resources
  CSV                  1 resources
  OTHER                1 resources
  GPKG                 1 resources
  SHP                  1 resources
  ARCGIS_REST          1 resources


In [ ]:
import geopandas as gpd
import requests

# Find a dataset with a WFS resource
wfs_dataset = None
wfs_url = None

for dataset in datasets:
    for resource in dataset.get('resources', []):
        if resource.get('format', '').upper() == 'WFS':
            wfs_dataset = dataset
            wfs_url = resource.get('url', '')
            break
    if wfs_dataset:
        break

if wfs_dataset:
    print(f"Found WFS dataset: {wfs_dataset['title']}")
    print(f"WFS URL: {wfs_url}\n")

    # Pull a small sample of features as GeoJSON
    # We add output format and limit parameters
    try:
        sample_url = wfs_url # Simplified from wfs_url.replace('wfs', 'wfs')

        # Build a proper WFS request
        wfs_params = {
            'service': 'WFS',
            'version': '2.0.0',
            'request': 'GetFeature',
            'outputFormat': 'json',
            'count': 5  # just 5 features to start
        }

        wfs_response = requests.get(wfs_url, params=wfs_params)
        print(f"Response status: {wfs_response.status_code}")
        print(f"First 500 chars of response:")
        print(wfs_response.text[:500])

    except Exception as e:
        print(f"Error: {e}")
else:
    print("No WFS dataset found in the search results. Cannot proceed with WFS request.")

No WFS dataset found in the search results. Cannot proceed with WFS request.


In [ ]:
def feature_to_llm_text(feature, dataset_title):
    """
    Convert a GeoJSON feature into structured text
    that an LLM can understand and reason about
    """
    props = feature.get('properties', {})
    geometry = feature.get('geometry', {})

    # Summarize geometry without raw coordinates
    geom_type = geometry.get('type', 'Unknown')
    coords = geometry.get('coordinates', [])

    # Build readable text
    lines = []
    lines.append(f"Dataset: {dataset_title}")
    lines.append(f"Feature Type: {geom_type}")

    # Add all properties as key-value pairs
    lines.append("\nAttributes:")
    for key, value in props.items():
        if value is not None and value != '':
            lines.append(f"  {key}: {value}")

    return "\n".join(lines)
